# OctoTetrahedral AGI — ARC-AGI-3 Agent v7
Robust HAVE_ARC/HAVE_AGENTS split, always writes submission.parquet.

In [ ]:
import subprocess, sys, os

COMP_DIR   = "/kaggle/input/arc-prize-2026-arc-agi-3"
WHEELS_DIR = os.path.join(COMP_DIR, "arc_agi_3_wheels")
AGENTS_DIR = os.path.join(COMP_DIR, "ARC-AGI-3-Agents")

# Install arcengine from competition wheels if available
if os.path.isdir(WHEELS_DIR):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index",
         "--find-links", WHEELS_DIR, "--quiet", "arcengine", "arc_agi"],
        capture_output=True, text=True)
    print("Wheel install:", r.returncode, r.stderr[:200] if r.returncode != 0 else "OK")

# Add agents path
if os.path.isdir(AGENTS_DIR) and AGENTS_DIR not in sys.path:
    sys.path.insert(0, AGENTS_DIR)

# Detect evaluation context — only require arcengine (agents is optional)
HAVE_ARC = False
try:
    import arcengine
    HAVE_ARC = True
    print("arcengine available — evaluation mode")
except ImportError as e:
    print(f"arcengine not available ({e}) — stub mode")

# Try agents module separately so partial import doesn't block us
HAVE_AGENTS = False
try:
    from agents.agent import Agent
    from agents import AVAILABLE_AGENTS, Swarm
    HAVE_AGENTS = True
    print("agents module available")
except (ImportError, Exception) as e:
    print(f"agents not available ({e})")

print(f"Setup done — HAVE_ARC={HAVE_ARC} HAVE_AGENTS={HAVE_AGENTS}")
if os.path.isdir(COMP_DIR):
    print("Comp dir contents:", os.listdir(COMP_DIR)[:10])


In [ ]:
import os, sys, random, logging, threading, time, json

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout)])
logger = logging.getLogger()

SCHEME  = os.environ.get("SCHEME", "http")
HOST    = os.environ.get("HOST",   "localhost")
PORT    = os.environ.get("PORT",   "8001")
ROOT_URL = f"{SCHEME}://{HOST}:{PORT}"
API_KEY  = os.environ.get("ARC_API_KEY", "")
HEADERS  = {"X-API-Key": API_KEY, "Accept": "application/json"}

OUT_FILE = "/kaggle/working/submission.parquet"

def _write_parquet(data: dict):
    os.makedirs("/kaggle/working", exist_ok=True)
    try:
        import pandas as pd
        pd.DataFrame([{k: str(v) for k, v in data.items()}]).to_parquet(OUT_FILE, index=False)
    except Exception as e:
        logger.warning(f"pandas parquet failed ({e}), trying pyarrow")
        try:
            import pyarrow as pa
            import pyarrow.parquet as pq
            table = pa.table({k: [str(v)] for k, v in data.items()})
            pq.write_table(table, OUT_FILE)
        except Exception as pe:
            fallback_json = "/kaggle/working/submission.json"
            with open(fallback_json, "w", encoding="utf-8") as f:
                json.dump(data, f, default=str)
            raise RuntimeError(
                f"Failed to write parquet with pandas ({e}) and pyarrow ({pe}). Wrote {fallback_json}."
            ) from pe
    logger.info(f"Wrote {OUT_FILE}")

if HAVE_ARC and HAVE_AGENTS:
    # Define our agent
    try:
        from arcengine import FrameData, GameAction, GameState

        class OctoTetrahedralAgent(Agent):
            def choose_action(self, game_state: GameState, frame: FrameData) -> GameAction:
                avail = [a for a in (game_state.available_actions or []) if a is not None]
                if not avail:
                    return GameAction()
                # Greedy: pick action that minimizes heuristic (random for now)
                return random.choice(avail)

        AVAILABLE_AGENTS["octotetrahedral_agi"] = OctoTetrahedralAgent
        logger.info(f"Agent registered. Available: {list(AVAILABLE_AGENTS.keys())}")
    except Exception as e:
        logger.error(f"Agent definition error: {e}")
        HAVE_AGENTS = False

# Discover games from server
games = []
if HAVE_ARC:
    import requests as _req
    logger.info(f"Connecting to game server at {ROOT_URL}")
    for attempt in range(20):
        try:
            r = _req.get(f"{ROOT_URL}/api/games", headers=HEADERS, timeout=5)
            if r.status_code == 200:
                games = [g["game_id"] for g in r.json()]
                logger.info(f"Server ready! Games: {games}")
                break
        except Exception:
            pass
        time.sleep(3)
    else:
        logger.warning("Game server not reachable")

logger.info(f"Games found: {len(games)}")

# Run agent if possible
scorecard_data = {"agent": "OctoTetrahedralAGI", "games_played": 0, "status": "stub"}

if HAVE_ARC and HAVE_AGENTS and games:
    try:
        swarm = Swarm("octotetrahedral_agi", ROOT_URL, games,
                      tags=["octotetrahedral-agi", "transcendplexity"])
        final_sc = None

        def run_swarm(sw):
            sw.main()

        def finalize_scorecard(sw):
            try:
                cid = getattr(sw, "card_id", None)
                if cid:
                    sc = sw.close_scorecard(cid)
                    if sc:
                        sw.cleanup(sc)
                        return sc
            except Exception as ex:
                logger.error(f"Cleanup: {ex}")
            return None

        t = threading.Thread(target=run_swarm, args=(swarm,), daemon=True)
        t.start()
        while t.is_alive():
            t.join(timeout=10)

        final_sc = finalize_scorecard(swarm)

        if final_sc:
            scorecard_data = final_sc.model_dump()
            scorecard_data["status"] = "complete"
        else:
            scorecard_data = {"agent": "OctoTetrahedralAGI", "games": str(games), "status": "finished"}

    except Exception as e:
        logger.error(f"Swarm error: {e}", exc_info=True)
        scorecard_data = {"agent": "OctoTetrahedralAGI", "error": str(e), "status": "error"}
else:
    logger.info("Not in evaluation context — writing stub submission")

_write_parquet(scorecard_data)
print("Done.", json.dumps(scorecard_data, default=str, indent=2)[:300])
